# Silver layer transformation and cleansing data notebook
Location of the data: _workspace.bronze.crm_sales_details_

1. Check data types - dates
2. Rename column headers
3. Trim data
4. Sales and Price inspection and correction

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length

## Read data:

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")
df.display()

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
print(df.dtypes)

## Clean data types for columns:
('sls_order_dt', 'int'), ('sls_ship_dt', 'int'), ('sls_due_dt', 'int')

In [0]:
df = (
    df
    .withColumn("sls_order_dt", F.try_to_date(F.col("sls_order_dt"), "yyyyMMdd"))
    .withColumn("sls_ship_dt", F.try_to_date(F.col("sls_ship_dt"), "yyyyMMdd"))
    .withColumn("sls_due_dt", F.try_to_date(F.col("sls_due_dt"), "yyyyMMdd"))
)

df.display()

## Sales and Price corrections

In [0]:
df = (
    df
    .withColumn(
        "sls_price",
        F.when(
            (col("sls_price").isNull()) | (col("sls_price") <= 0),
            F.when(
                col("sls_quantity") != 0,
                col("sls_sales") / col("sls_quantity")
            ).otherwise(None)
        ).otherwise(col("sls_price"))
    )
)

## Rename columns

In [0]:
rename_map = {
    "sls_ord_num": "order_id",
    "sls_prd_key": "product_id",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

for old, new in rename_map.items():
    df = df.withColumnRenamed(old, new)

df.display()

## Write to silver catalog as delta table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")